# ==========================================================
# Breast Cancer Survival Prediction using Apache Spark
# Notebook 03: Feature Engineering
# ==========================================================

Objective
---------
1. Create the target variable for survival prediction.
2. Select relevant features for machine learning.
3. Encode categorical variables using Spark ML.
4. Assemble features into a feature vector.
5. Split the dataset into training and testing sets.
6. Prepare the final dataset for model training.

Note
----
No data preprocessing.
No model training.
No model evaluation.


In [1]:
# 1. Import Libraries

import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)

from pyspark.sql.types import *

In [2]:
# 2. Create Spark Session

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.data.loader import (
    create_spark_session,
    load_csv
)

spark = create_spark_session(
    "SEER Breast Cancer Feature Engineering"
)

In [ ]:
# 3. Load Clean Dataset

df = load_csv(
    spark,
    "../data/processed/seer_breast_cancer_clean.csv"
)

# Store dataset information

dataset_row_count = df.count()
dataset_column_count = len(df.columns)

CLEAN DATASET INFORMATION
Rows    : 456,087
Columns : 25


In [4]:
# 4. Dataset Overview

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print(f"Number of Rows    : {dataset_row_count:,}")
print(f"Number of Columns : {dataset_column_count}")

DATASET OVERVIEW
Number of Rows    : 456,087
Number of Columns : 25


In [5]:
# 5. Schema

print("=" * 60)
print("DATASET SCHEMA")
print("=" * 60)

df.printSchema()

DATASET SCHEMA
root
 |-- Age: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Race: string (nullable = true)
 |-- Marital_Status: string (nullable = true)
 |-- Tumor_Size: double (nullable = true)
 |-- Survival_Months: double (nullable = true)
 |-- Vital_Status: string (nullable = true)
 |-- Grade: string (nullable = true)
 |-- PR_Status: string (nullable = true)
 |-- ER_Status: string (nullable = true)
 |-- AJCC_T: string (nullable = true)
 |-- AJCC_N: string (nullable = true)
 |-- Regional_Nodes_Examined: double (nullable = true)
 |-- Regional_Nodes_Positive: double (nullable = true)
 |-- Sequence_Number: string (nullable = true)
 |-- Histologic_Type: integer (nullable = true)
 |-- Laterality: string (nullable = true)
 |-- Diagnostic_Confirmation: string (nullable = true)
 |-- AJCC_M: string (nullable = true)
 |-- Surgery_Primary_Site: integer (nullable = true)
 |-- Surgery_Other_Regional: string (nullable = true)
 |-- Surgery_Radiation_Sequence: string (nullable = t

In [6]:
# 6. Preview Dataset

print("=" * 60)
print("DATASET PREVIEW")
print("=" * 60)

df.show(10, truncate=False)

DATASET PREVIEW


+-----------+------+-------------------------+------------------------------+----------+---------------+------------+-----------------------------------+---------+---------+-----------+-----------+-----------------------+-----------------------+----------------+---------------+-------------------------+-----------------------+------+--------------------+--------------------------+-------------------------------------------------------------------------+------------------------------------+------------+----------+
|Age        |Sex   |Race                     |Marital_Status                |Tumor_Size|Survival_Months|Vital_Status|Grade                              |PR_Status|ER_Status|AJCC_T     |AJCC_N     |Regional_Nodes_Examined|Regional_Nodes_Positive|Sequence_Number |Histologic_Type|Laterality               |Diagnostic_Confirmation|AJCC_M|Surgery_Primary_Site|Surgery_Other_Regional    |Surgery_Radiation_Sequence                                               |Radiation               

In [7]:
# 7. Create Target Variable

print("=" * 60)
print("CREATING TARGET VARIABLE")
print("=" * 60)

# Create binary target variable
# Alive = 0
# Dead = 1

df = df.withColumn(
    "label",
    F.when(
        F.col("Vital_Status") == "Dead",
        1
    ).otherwise(0)
)

print("Target variable 'label' created successfully.")
print("Alive -> 0")
print("Dead  -> 1")

CREATING TARGET VARIABLE
Target variable 'label' created successfully.
Alive -> 0
Dead  -> 1


In [8]:
# 8. Verify Target Variable

print("=" * 60)
print("TARGET VARIABLE VALIDATION")
print("=" * 60)

df.groupBy(
    "Vital_Status",
    "label"
).count() \
.orderBy("label") \
.show(truncate=False)

print()

print("Label Distribution")
print("-" * 60)

df.groupBy("label") \
.count() \
.orderBy("label") \
.show()

TARGET VARIABLE VALIDATION


+------------+-----+------+
|Vital_Status|label|count |
+------------+-----+------+
|Alive       |0    |287306|
|Dead        |1    |168781|
+------------+-----+------+


Label Distribution
------------------------------------------------------------
+-----+------+
|label| count|
+-----+------+
|    0|287306|
|    1|168781|
+-----+------+



In [9]:
# 9. Feature Selection

print("=" * 60)
print("FEATURE SELECTION")
print("=" * 60)

# Features used for machine learning

feature_columns = [
    "Age",
    "Sex",
    "Race",
    "Marital_Status",
    "Tumor_Size",
    "Grade",
    "PR_Status",
    "ER_Status",
    "AJCC_T",
    "AJCC_N",
    "AJCC_M",
    "AJCC_Stage",
    "Regional_Nodes_Examined",
    "Regional_Nodes_Positive",
    "Histologic_Type",
    "Laterality",
    "Diagnostic_Confirmation",
    "Surgery_Primary_Site",
    "Surgery_Other_Regional",
    "Surgery_Radiation_Sequence",
    "Radiation",
    "Chemotherapy"
]

print(f"Selected Features : {len(feature_columns)}")
print()

for column in feature_columns:
    print(column)

FEATURE SELECTION
Selected Features : 22

Age
Sex
Race
Marital_Status
Tumor_Size
Grade
PR_Status
ER_Status
AJCC_T
AJCC_N
AJCC_M
AJCC_Stage
Regional_Nodes_Examined
Regional_Nodes_Positive
Histologic_Type
Laterality
Diagnostic_Confirmation
Surgery_Primary_Site
Surgery_Other_Regional
Surgery_Radiation_Sequence
Radiation
Chemotherapy


In [10]:
# 10. StringIndexer

print("=" * 60)
print("STRING INDEXING")
print("=" * 60)

from pyspark.ml.feature import StringIndexer

# Identify categorical features

categorical_columns = [

    "Age",
    "Sex",
    "Race",
    "Marital_Status",
    "Grade",
    "PR_Status",
    "ER_Status",
    "AJCC_T",
    "AJCC_N",
    "AJCC_M",
    "AJCC_Stage",
    "Laterality",
    "Diagnostic_Confirmation",
    "Surgery_Other_Regional",
    "Surgery_Radiation_Sequence",
    "Radiation",
    "Chemotherapy"
]

indexers = [
    StringIndexer(
        inputCol=column,
        outputCol=f"{column}_Index",
        handleInvalid="keep"
    )
    for column in categorical_columns
]

print(f"Categorical Features : {len(categorical_columns)}")
print(f"StringIndexers       : {len(indexers)}")

STRING INDEXING
Categorical Features : 17
StringIndexers       : 17


In [11]:
# 11. Verify Indexed Features

print("=" * 60)
print("VERIFY INDEXED FEATURES")
print("=" * 60)

pipeline = Pipeline(
    stages=indexers
)

index_model = pipeline.fit(df)
df_indexed = index_model.transform(df)

print("Indexed Columns")
print("-" * 60)

for column in categorical_columns:
    print(f"{column:<35} --> {column}_Index")
print()

df_indexed.select(
    "Race",
    "Race_Index",
    "Grade",
    "Grade_Index",
    "AJCC_Stage",
    "AJCC_Stage_Index"

).show(10, truncate=False)

VERIFY INDEXED FEATURES
Indexed Columns
------------------------------------------------------------
Age                                 --> Age_Index
Sex                                 --> Sex_Index
Race                                --> Race_Index
Marital_Status                      --> Marital_Status_Index
Grade                               --> Grade_Index
PR_Status                           --> PR_Status_Index
ER_Status                           --> ER_Status_Index
AJCC_T                              --> AJCC_T_Index
AJCC_N                              --> AJCC_N_Index
AJCC_M                              --> AJCC_M_Index
AJCC_Stage                          --> AJCC_Stage_Index
Laterality                          --> Laterality_Index
Diagnostic_Confirmation             --> Diagnostic_Confirmation_Index
Surgery_Other_Regional              --> Surgery_Other_Regional_Index
Surgery_Radiation_Sequence          --> Surgery_Radiation_Sequence_Index
Radiation                           --

In [12]:
# 12. OneHotEncoder

print("=" * 60)
print("ONE-HOT ENCODING")
print("=" * 60)

from pyspark.ml.feature import OneHotEncoder

encoder = OneHotEncoder(
    inputCols=[f"{column}_Index"for column in categorical_columns],
    outputCols=[f"{column}_Vec"for column in categorical_columns]
)

encoder_model = encoder.fit(df_indexed)
df_encoded = encoder_model.transform(df_indexed)

print(f"Encoded Features : {len(categorical_columns)}")

ONE-HOT ENCODING
Encoded Features : 17


In [13]:
# 13. Verify Encoded Features

print("=" * 60)
print("VERIFY ENCODED FEATURES")
print("=" * 60)

df_encoded.select(

    "Race",
    "Race_Vec",

    "Grade",
    "Grade_Vec",

    "AJCC_Stage",
    "AJCC_Stage_Vec"

).show(10, truncate=False)

VERIFY ENCODED FEATURES
+-------------------------+-------------+-----------------------------------+-------------+----------+--------------+
|Race                     |Race_Vec     |Grade                              |Grade_Vec    |AJCC_Stage|AJCC_Stage_Vec|
+-------------------------+-------------+-----------------------------------+-------------+----------+--------------+
|White                    |(5,[0],[1.0])|Moderately differentiated; Grade II|(5,[0],[1.0])|IIA       |(10,[1],[1.0])|
|White                    |(5,[0],[1.0])|Moderately differentiated; Grade II|(5,[0],[1.0])|IIB       |(10,[2],[1.0])|
|White                    |(5,[0],[1.0])|Moderately differentiated; Grade II|(5,[0],[1.0])|I         |(10,[0],[1.0])|
|White                    |(5,[0],[1.0])|Moderately differentiated; Grade II|(5,[0],[1.0])|IIIC      |(10,[6],[1.0])|
|White                    |(5,[0],[1.0])|Well differentiated; Grade I       |(5,[2],[1.0])|I         |(10,[0],[1.0])|
|Black                    |(5,[1

In [14]:
# 14. Handle Missing Numerical Features

print("=" * 60)
print("HANDLE NUMERICAL MISSING VALUES")
print("=" * 60)

numeric_missing_features = [
    "Tumor_Size",
    "Regional_Nodes_Examined",
    "Regional_Nodes_Positive"
]


for col in numeric_missing_features:
    median_value = (df_encoded.approxQuantile(col,[0.5],0.01)[0])

    print(f"{column:<35} Median = {median_value}")
    df_encoded = df_encoded.fillna({col: median_value})

print()
print("Missing numerical values handled successfully.")

HANDLE NUMERICAL MISSING VALUES
Chemotherapy                        Median = 18.0
Chemotherapy                        Median = 3.0
Chemotherapy                        Median = 0.0

Missing numerical values handled successfully.


In [15]:
# 15. Verify Missing Values

print("=" * 60)
print("VERIFY NUMERICAL MISSING VALUES")
print("=" * 60)

for column in numeric_missing_features:
    null_count = (df_encoded.filter(F.col(column).isNull()).count())
    print(f"{column:<35} Null = {null_count}")

VERIFY NUMERICAL MISSING VALUES
Tumor_Size                          Null = 0
Regional_Nodes_Examined             Null = 0
Regional_Nodes_Positive             Null = 0


In [17]:
# 16. Feature Metadata

print("="*60)
print("FEATURE METADATA")
print("="*60)

# Encoded feature groups

encoded_features = [
    "Age_Vec",
    "Sex_Vec",
    "Race_Vec",
    "Marital_Status_Vec",
    "Grade_Vec",
    "PR_Status_Vec",
    "ER_Status_Vec",
    "AJCC_T_Vec",
    "AJCC_N_Vec",
    "AJCC_M_Vec",
    "AJCC_Stage_Vec",
    "Laterality_Vec",
    "Diagnostic_Confirmation_Vec",
    "Surgery_Other_Regional_Vec",
    "Surgery_Radiation_Sequence_Vec",
    "Radiation_Vec",
    "Chemotherapy_Vec"
]

# Numerical Features

numeric_features = [
    "Tumor_Size",
    "Regional_Nodes_Examined",
    "Regional_Nodes_Positive",
    "Histologic_Type",
    "Surgery_Primary_Site"
]

# Final Feature Columns

feature_columns = (encoded_features + numeric_features)

print(f"Encoded Feature Groups : {len(encoded_features)}")
print(f"Numeric Features       : {len(numeric_features)}")
print(f"Total Feature Groups   : {len(feature_columns)}")

# Feature Mapping

feature_mapping = {index:feature for index,feature in enumerate(feature_columns)}

print("\nFeature Mapping")
print("="*60)

for index, feature in feature_mapping.items():
    print(f"{index:03d} -> {feature}")

FEATURE METADATA
Encoded Feature Groups : 17
Numeric Features       : 5
Total Feature Groups   : 22

Feature Mapping
000 -> Age_Vec
001 -> Sex_Vec
002 -> Race_Vec
003 -> Marital_Status_Vec
004 -> Grade_Vec
005 -> PR_Status_Vec
006 -> ER_Status_Vec
007 -> AJCC_T_Vec
008 -> AJCC_N_Vec
009 -> AJCC_M_Vec
010 -> AJCC_Stage_Vec
011 -> Laterality_Vec
012 -> Diagnostic_Confirmation_Vec
013 -> Surgery_Other_Regional_Vec
014 -> Surgery_Radiation_Sequence_Vec
015 -> Radiation_Vec
016 -> Chemotherapy_Vec
017 -> Tumor_Size
018 -> Regional_Nodes_Examined
019 -> Regional_Nodes_Positive
020 -> Histologic_Type
021 -> Surgery_Primary_Site


In [18]:
# 17. VectorAssembler

print("=" * 60)
print("VECTOR ASSEMBLER")
print("=" * 60)

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features", handleInvalid="keep")

df_features = assembler.transform(df_encoded)

print("Feature vector created successfully.")

VECTOR ASSEMBLER
Feature vector created successfully.


In [ ]:
# 18. Verify Feature Vector
print("=" * 60)
print("VERIFY FEATURE VECTOR")
print("=" * 60)

df_features.select("features","label").show(5, truncate=False)

print()

print("Feature Vector Schema")
print("-" * 60)

df_features.select("features").printSchema()

VERIFY FEATURE VECTOR
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                                                                        |label|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|(120,[2,19,21,26,33,38,42,46,60,64,68,78,82,90,98,105,114,115,116,117,118,119],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,15.0,7.0,1.0,8522.0,22.0])  |0    |
|(120,[2,19,21,26,33,38,42,52,59,64,69,77,82,90,97,109,114,115,116,118,119],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,83.0,1.0,8500.0,75.0])          |0    |
|(120,[7,19,21,26,33,38,42,46,59,64,67,77,82,90,97,10

In [ ]:
# 19. Train/Test Split

print("=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)

print(f"Training : {train_df.count():,}")

print(f"Testing  : {test_df.count():,}")

TRAIN / TEST SPLIT
Training : 365,322
Testing  : 90,765


In [ ]:
# 20. Check Class Distribution

print("Training")

train_df.groupBy("label").count().show()

print("Testing")
test_df.groupBy("label").count().show()

Training
+-----+------+
|label| count|
+-----+------+
|    1|135282|
|    0|230040|
+-----+------+

Testing
+-----+-----+
|label|count|
+-----+-----+
|    1|33499|
|    0|57266|
+-----+-----+



In [ ]:
# 21. Handle Class Imbalance

print("="*60)
print("CLASS IMBALANCE")
print("="*60)

class_distribution = (train_df.groupBy("label").count())
class_distribution.show()

print()
print("No class balancing method applied.")


print(""" Current strategy:
- Use original class distribution
- No oversampling
- No undersampling
- No SMOTE

Future improvement:
- classWeightCol
- Random oversampling
- Undersampling
""")

CLASS IMBALANCE
+-----+------+
|label| count|
+-----+------+
|    1|135282|
|    0|230040|
+-----+------+

No resampling applied.
Original distribution will be used.


In [ ]:
# 22. Validate Feature Dataset

print(f"Rows    : {df_features.count():,}")
print(f"Columns : {len(df_features.columns)}")

df_features.select("features","label").show(5,truncate=False)
print()

print("Schema")
df_features.select("features", "label").printSchema()


Rows    : 456,087
Columns : 61
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                                                                        |label|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|(120,[2,19,21,26,33,38,42,46,60,64,68,78,82,90,98,105,114,115,116,117,118,119],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,15.0,7.0,1.0,8522.0,22.0])  |0    |
|(120,[2,19,21,26,33,38,42,52,59,64,69,77,82,90,97,109,114,115,116,118,119],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,83.0,1.0,8500.0,75.0])          |0    |
|(120,[7,19,21,26,33,38,42,46,59,64,67,77,82

In [ ]:
# 23. Feature Engineering Report

print(f"""
      
Target Variable

--------------------------------------------------
label
0 -> Alive
1 -> Dead
      
Feature Engineering
--------------------------------------------------
Selected Feature Groups : {len(feature_columns)}

Categorical Features    : {len(categorical_columns)}

Encoded Features        : {len(encoded_features)}

Numeric Features        : {len(numeric_features)}

Feature Vector
--------------------------------------------------
Output Column           : features

Dataset Split
--------------------------------------------------
Training Records        : {train_df.count():,}

Testing Records         : {test_df.count():,}

Missing Value Treatment
--------------------------------------------------

Numerical missing values:

- Tumor_Size
- Regional_Nodes_Examined
- Regional_Nodes_Positive

Method                  : Median Imputation

""")

Feature Engineering Report

Dataset: SEER Breast Cancer

Selected Feature Groups:
22

Categorical Features:
17

Encoded Features:
17

Numeric Features:
5

Training Records:
365,322

Testing Records:
90,765

Target:
label

0 = Alive
1 = Dead

Status:
Completed successfully.



In [ ]:
# 24. Export Feature Dataset (Optional)

print("=" * 60)
print("EXPORT FEATURE DATASET")
print("=" * 60)


print(""" Optional step.

The feature dataset is kept in memory.
Notebook 04 — Model Training
      
will directly use:
- train_df
- test_df
""")

Feature dataset kept in memory.
Notebook 04 will directly use train_df and test_df.


In [ ]:
# 25. Feature Engineering Summary

print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)

print(f""" Feature Engineering Completed Successfully.

Dataset: SEER Breast Cancer

Final Feature Vector: features

Target: label

Data Split
------------------------------------------------------------
Training Dataset    : {train_df.count():,}

Testing Dataset     : {test_df.count():,}

Class Imbalance
------------------------------------------------------------
No resampling applied.

Output
------------------------------------------------------------
Feature dataset is ready for:

Notebook 04 — Model Training

Status
------------------------------------------------------------
✓ Feature engineering completed successfully.


============================================================

""")





FEATURE ENGINEERING SUMMARY



Dataset:

SEER Breast Cancer


Feature Groups:

22


Categorical:

17


Encoded:

17


Numeric:

5



Train Dataset:

365,322


Test Dataset:

90,765



Output:

Ready for Notebook 04 — Model Training




